# Interactive Dashboards &mdash; `nb.dashboard()`

`UnichartNotebook` plotting methods each return a Plotly figure. The companion module
`unichart_dashboard` wires several of them into a single **interactive Dash board** that
renders inline in this notebook.

The board shows **one consistent slice of the data**: a header bar owns the dataset
selection and the light/dark theme for every panel, and each panel keeps only the
controls that are genuinely its own.

**Header (applies to the whole board):**

| Control | Effect |
|---|---|
| **datasets** | toggle which sets every panel plots (chips show each set's plot color) |
| **theme** | light / dark &mdash; restyles the chrome *and* every figure together |
| **⬇ export all panels** | download every panel's data as one CSV |

**Per panel:**

| Control | Effect |
|---|---|
| **title** | the card title, editable in place (also names the CSV export) |
| **plot type** | switch the panel between `plot`, `plot_ymult`, `bar`, `box`, `histogram`, `contour`, `table` |
| **x** | choose the x column |
| **y** | choose one or more y columns (multi-select) |
| **z** | contour color column (shown only for `contour`) |
| **legend** | `above` / `right` / `off` (shown only for `plot` / `plot_ymult`) |
| **⬇ csv** | download this panel's plotted data |

Changing a panel control re-plots **only that panel**; changing the header's datasets or
theme re-plots the whole board, reusing all of the notebook's formatting (axis limits,
variable formats, lines, highlights).

This notebook covers the live board (§ 1&ndash;2), the options (§ 3), a locked-down
presentation board (§ 4, `controls=False`), and exporting a board to a standalone HTML
file (§ 5, `uc.dashboard_to_html`).

> Requires Dash (`pip install dash`). It is an optional dependency &mdash; the rest of the
> toolkit imports fine without it. (The § 5 HTML export needs only Plotly, not Dash.)

## 0. Setup &mdash; two sensor-log datasets

Two runs of a rig logged over a `time` sweep, with continuous channels (`pressure`,
`temp`, `flow`) and a categorical `phase`. The mix of continuous and categorical columns
lets the same board show line plots, bars, boxes, and histograms.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(0)

def run(scale):
    t = np.arange(0, 60)
    phase = np.where(t < 15, 'idle', np.where(t < 40, 'ramp', 'peak'))
    return pd.DataFrame({
        'time':     t,
        'pressure': scale * (10 + 0.8 * t + rng.normal(0, 1.5, t.size)),
        'temp':     scale * (300 + 6 * t + rng.normal(0, 8, t.size)),
        'flow':     scale * (5 + 0.3 * t + rng.normal(0, 0.5, t.size)),
        'phase':    phase,
    })

uc = UnichartNotebook()
uc.load(run(1.00), title='Run A')   # Set 0
uc.load(run(1.15), title='Run B')   # Set 1
uc.sets[0].df.head()

UniChart Notebook Environment Initialized.
Loaded Set 0: Run A
Loaded Set 1: Run B


,time,pressure,temp,flow,phase
0,0,10.188595,296.508518,5.393794,idle
1,1,10.601843,296.641585,5.722039,idle
2,2,12.560634,325.914943,5.637797,idle
3,3,12.557350,314.032714,5.186613,idle
4,4,12.396496,326.631757,6.132477,idle


## 1. Minimal board

Pass the notebook and a list of **panel specs**. Each spec seeds that panel's initial
controls; `method` defaults to `'plot'`. The call launches a Dash server and renders it
inline (`jupyter_mode='inline'`).

Try it: untick a **dataset** in the header and both panels drop it together; flip the
**theme** to dark and the whole board &mdash; chrome and figures &mdash; follows. Then change one
panel's **plot type** or **x/y** and watch just that panel update.

In [2]:
uc.dashboard([
    {'method': 'plot', 'x': 'time', 'y': 'pressure'},
    {'method': 'plot', 'x': 'time', 'y': 'temp'},
])

## 2. A richer board

Panels can mix plot types and seed any control. `y` accepts a list (multi-select; useful
for `plot_ymult`). A panel with a `datasets` key is **pinned** to those sets: it ignores
the header picker and carries a "pinned" badge &mdash; here the box panel always shows only
Run A, whatever the header selects.

`title` names the board, `ncols` sets the grid width, and `height` the px height of each
panel's graph.

In [3]:
uc.dashboard(
    panels=[
        {'method': 'plot',      'x': 'time',  'y': ['pressure', 'temp'], 'suptitle': 'Channels vs time'},
        {'method': 'plot_ymult','x': 'time',  'y': ['pressure', 'flow'], 'legend': 'right'},
        {'method': 'bar',       'x': 'phase', 'y': 'flow',  'suptitle': 'Mean flow by phase'},
        {'method': 'box',       'x': 'phase', 'y': 'temp',  'datasets': [0]},   # pinned to Run A
        {'method': 'histogram', 'x': 'temp',  'y': 'temp'},
    ],
    title='Rig overview',
    ncols=2,
    height=380,
)

## 3. Options & notes

- **Run target.** `jupyter_mode='inline'` (default) renders in the cell. Use
  `jupyter_mode='external'` (or `'tab'`) to open it in a browser instead, e.g.
  `uc.dashboard(panels, jupyter_mode='external', port=8051)`.
- **Plot types.** The live switch covers `plot`, `plot_ymult`, `bar`, `box`, `histogram`,
  `contour` (needs a `z` column &mdash; its dropdown appears when selected) and `table`
  (the `y` control picks the columns to show).
- **`legend`** applies to `plot` / `plot_ymult` only; its control is hidden for the
  other types, as is `z` outside `contour`.
- **Titles live in the card.** A panel's title is the editable text at the top of its
  card (chrome), not a figure suptitle &mdash; so the plot area stays clean and the title
  also names the panel's CSV export.
- **Theme.** The header switch seeds from `uc.darkmode` and restyles chrome + figures
  together; it never changes the notebook's own `darkmode` setting.
- **Shared state.** The header picker (or a panel's pin) flips each set's `.select` just
  for that render and restores it afterward, so the board never mutates your notebook.
  Renders are serialized with a lock, so keep the board to a single user / kernel.
- **Errors** in a panel (e.g. an empty `y`, or no datasets selected) show as a message
  inside that panel rather than crashing the whole board.
- **Export data.** Each panel's **⬇ csv** button downloads its plotted data as CSV
  (tidy `trace, x, y[, z]`; contour exports the surface grid plus overlay points). The
  header's **⬇ export all panels** button combines every panel into one CSV with a
  `panel` column.
- **Locked / presentation mode.** Pass `controls=False` for a locked-down board: the
  per-panel dropdowns, export buttons, header dataset picker and theme switch, and
  title editing are all hidden, leaving a clean grid of titled figure cards that render
  exactly the seeded panels. Good for reports, a wall display, or handing someone a
  fixed view. See § 4.

### Under the hood

`uc.dashboard(...)` is a thin wrapper around `unichart_dashboard.dashboard(uc, panels)`.
For testing or custom embedding you can build the app without launching a server with
`unichart_dashboard.build_app(uc, panels)`, or render a single panel to a figure with
`unichart_dashboard.render_panel(uc, 'plot', 'time', ['pressure'], dataset_indices=[0, 1])`.

## 4. Locked / presentation board

The same panels with `controls=False`. Every editing affordance is gone &mdash; no
per-panel dropdowns, no export buttons, no header picker or theme switch, and the titles
are display-only. The board still renders exactly the seeded panels (and honors the
box panel's pin to Run A), so it's a clean fixed view for a report or a wall display.

Seed the theme with `uc.toggle_darkmode(True)` before the call if you want it to open
dark (the switch is hidden, so it can't be flipped afterward).

In [4]:
uc.dashboard(
    panels=[
        {'method': 'plot',      'x': 'time',  'y': ['pressure', 'temp'], 'suptitle': 'Channels vs time'},
        {'method': 'plot_ymult','x': 'time',  'y': ['pressure', 'flow'], 'legend': 'right', 'suptitle': 'Pressure & flow'},
        {'method': 'bar',       'x': 'phase', 'y': 'flow',  'suptitle': 'Mean flow by phase'},
        {'method': 'box',       'x': 'phase', 'y': 'temp',  'datasets': [0], 'suptitle': 'Temp spread (Run A)'},
    ],
    title='Rig overview',
    controls=False,   # locked-down presentation board
    ncols=2,
    height=360,
)

## 5. Export to a static HTML file

A live board is a Dash **server** &mdash; there's nothing to "Save Page As". To hand someone
a file, use `uc.dashboard_to_html(panels, path)`: it renders each panel once and writes
the board's card grid to a single HTML page. The Plotly charts stay interactive
(hover / zoom / pan / modebar); only the editing chrome (dropdowns, picker, theme switch)
is dropped &mdash; it's the locked view frozen to disk.

It's a **fixed snapshot** of the notebook's current dataset selection and theme, so seed
those first if you want a specific slice (`uc.toggle_darkmode(True)`, or set each
`uc.sets[i].select`). Pinned panels (`datasets=[...]`) still honor their pin.

The one knob is **`embed_js`** &mdash; how `plotly.js` is bundled:

| `embed_js` | File | Opens offline? |
|---|---|---|
| `'cdn'` *(default)* | small (~200 KB) | needs internet |
| `True` | self-contained (~5 MB) | yes &mdash; email / air-gapped |
| `'directory'` | small + a `plotly.min.js` beside it | yes, if the js file travels with it |

In [5]:
panels = [
    {'method': 'plot',      'x': 'time',  'y': ['pressure', 'temp'], 'suptitle': 'Channels vs time'},
    {'method': 'plot_ymult','x': 'time',  'y': ['pressure', 'flow'], 'legend': 'right', 'suptitle': 'Pressure & flow'},
    {'method': 'bar',       'x': 'phase', 'y': 'flow',  'suptitle': 'Mean flow by phase'},
    {'method': 'box',       'x': 'phase', 'y': 'temp',  'datasets': [0], 'suptitle': 'Temp spread (Run A)'},
]

# self-contained file you can email or open offline
path = uc.dashboard_to_html(panels, 'rig_overview.html',
                            title='Rig overview', embed_js=True)
print('wrote', path)


wrote rig_overview.html
